# EXP-025 — Qwen/CLIP semantic complementarity (research only)

Upload the existing TASK-025C `exp023_handoff.zip` to MyDrive, select an **L4 CUDA GPU**, set the one authorization below, and **Runtime → Run all**. No new data bundle is required. This uses pinned Python3.12.10 and dependencies; model downloads/images stay in /content and completed per-image semantic records/logs remain on Drive.

The local NF4 smoke had incorrect details and nationality wording. This notebook makes a **separate, unquantized FP16 attempt**; its quality and throughput are unverified. It stops and exports if smoke checks fail. If they pass it extracts all1335 P2 training paintings, evaluates two folds with matched visual/semantic/fusion/shuffle heads, and adds the remaining folds only under the frozen exploratory continuation rule. No captions, confirmation annotations, Development labels, E50 blending, fine-tuning or submission. Built with Qwen under its non-commercial Research License; competition eligibility is unresolved.

Defaults allow at most2 hours for an explicitly authorized run, not a target or universal cap. Long downloads count. An interrupted image may restart; completed image records persist. Head folds resume only with matching source/config/cache checksums. Export/download/release API calls do not prove browser arrival or physical GPU deallocation. No claim is made that FP16 fixes hallucinations.


In [ ]:
# OWNER CONFIGURATION: set authorization once, then Runtime -> Run all.
AUTHORIZE_EXP025 = False
INPUT_ZIP = "/content/drive/MyDrive/exp023_handoff.zip"
PERSISTENT_ROOT = "/content/drive/MyDrive/exp025"
LOCAL_ROOT = "/content/exp025"
MAX_HOURS = 2.0
AUTO_DOWNLOAD = True
AUTO_RELEASE = True
SOURCE_REVISION = "6cb68f9a52c46f4aca639b1dcd7be5977f295727"
ZIP_SHA256 = "2db4a53679940c7eb63c7c8272b91be8ec0618bb6c04a69ebaa15909e20f64e0"


In [ ]:
import csv, hashlib, json, os, shutil, subprocess, sys, time, traceback, zipfile
from pathlib import Path
from google.colab import drive, files, runtime
started = time.time()
local, persistent = Path(LOCAL_ROOT), Path(PERSISTENT_ROOT)
attempt = persistent / "attempts" / time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
result = {"status":"STARTING", "research_only":True, "production":"E50+L unchanged",
          "browser_arrival":"UNVERIFIED", "physical_GPU_deallocation":"UNVERIFIED"}
mounted = False

def run(command, name, env=None):
    remaining = started + MAX_HOURS*3600 - time.time() - 120
    if remaining <= 0: raise TimeoutError("Allocation exhausted; completed semantic records persist")
    print("Running", name, flush=True)
    log = attempt / (name + ".log")
    with log.open("w") as handle:
        process = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT, timeout=remaining, env=env)
    tail = log.read_text(errors="replace").splitlines()[-16:]
    print("\n".join(tail))
    if process.returncode: raise RuntimeError(f"{name} failed; see {log}. Cached completed images are retained.")

try:
    drive.mount("/content/drive");mounted = True
    attempt.mkdir(parents=True, exist_ok=False);local.mkdir(parents=True, exist_ok=True)
    if not AUTHORIZE_EXP025: raise RuntimeError("Set AUTHORIZE_EXP025=True in Configuration before the authorized Run all")
    if not (0 < MAX_HOURS < float("inf")): raise ValueError("MAX_HOURS must be positive and finite")
    with Path(INPUT_ZIP).open("rb") as handle:
        if hashlib.file_digest(handle,"sha256").hexdigest() != ZIP_SHA256: raise ValueError("Use the exact TASK-025C private input ZIP; SHA256 mismatch")
    source = local / "source"
    if not source.exists():
        run(["git","clone","--no-checkout","https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git",str(source)],"clone")
        run(["git","-C",str(source),"checkout","--detach",SOURCE_REVISION],"checkout")
    if subprocess.check_output(["git","-C",str(source),"rev-parse","HEAD"],text=True).strip()!=SOURCE_REVISION:
        raise ValueError("Existing local source revision differs; choose a fresh LOCAL_ROOT")
    expected = json.loads((source/"configs/exp024.json").read_text())["input_sha256"]
    inputs = local / "inputs"
    with zipfile.ZipFile(INPUT_ZIP) as archive:
        protocol = archive.read("training/protocol.csv").decode("utf-8")
        rows = [row for row in csv.DictReader(protocol.splitlines()) if row["role"]=="development"]
        if len(rows)!=1335: raise ValueError("Expected1335 P2 training paintings")
        targets = {**expected, **{"artwork/images/"+row["image"]:row["image_sha256"] for row in rows}}
        for name, checksum in targets.items():
            target = (inputs/name).resolve()
            if not target.is_relative_to(inputs.resolve()): raise ValueError("Unsafe input ZIP path")
            if target.is_file():
                with target.open("rb") as handle: current=hashlib.file_digest(handle,"sha256").hexdigest()
                if current!=checksum: raise ValueError("Existing local input changed")
                continue
            raw = archive.read(name)
            if hashlib.sha256(raw).hexdigest()!=checksum: raise ValueError("Input checksum mismatch: "+name)
            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
    env = dict(os.environ, UV_PYTHON_INSTALL_DIR=str(local/"python"),UV_CACHE_DIR=str(local/"uv_cache"),HF_HUB_DISABLE_XET="1")
    run([sys.executable,"-m","pip","install","uv==0.7.13"],"uv",env)
    run([sys.executable,"-m","uv","python","install","3.12.10"],"python",env)
    if not (local/"venv").exists():run([sys.executable,"-m","uv","venv","--python","3.12.10",str(local/"venv")],"venv",env)
    python = str(local/"venv/bin/python")
    run([sys.executable,"-m","uv","pip","sync","--python",python,"--extra-index-url","https://download.pytorch.org/whl/cu126","--index-strategy","unsafe-best-match",str(source/"requirements-exp025.lock.txt")],"dependencies",env)
    run([sys.executable,"-m","uv","pip","check","--python",python],"dependency_check",env)
    run([python,"-c","import torch; assert torch.cuda.is_available(), 'Select CUDA GPU'; print(torch.cuda.get_device_name()); assert torch.cuda.mem_get_info()[0]>10*1024**3, 'Select L4 with at least10GiB free VRAM'"],"GPU_check",env)
    cache = persistent / "semantics"
    remaining = max(1,started+MAX_HOURS*3600-time.time()-120)
    run([python,str(source/"scripts/extract_exp025.py"),"--inputs",str(inputs),"--cache",str(cache),"--model-cache",str(local/"models"),"--full","--max-seconds",str(remaining)],"extraction",env)
    run([python,str(source/"scripts/run_exp025.py"),"--inputs",str(inputs),"--cache",str(cache),"--run-dir",str(persistent/"heads_fp16")],"heads",env)
    result["status"]="AUTHORIZED_PIPELINE_COMPLETE"
except Exception as error:
    result.update(status="STOPPED_FOR_REVIEW",error=str(error),traceback=traceback.format_exc())
finally:
    try:
        if mounted:
            attempt.mkdir(parents=True,exist_ok=True)
            (attempt/"status.json").write_text(json.dumps(result,indent=2))
            for report in (persistent/"semantics").glob("*/extraction_report.json"):
                print("Measured extraction report:",report.read_text())
            head_summary = persistent/"heads_fp16/summary.csv"
            if head_summary.exists():print("Head results:\n",head_summary.read_text())
            output = attempt/"outputs.zip"
            with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED) as archive:
                for directory,prefix in [(attempt,"attempt"),(persistent/"semantics","semantics"),(persistent/"heads_fp16","heads")]:
                    if directory.exists():
                        for path in directory.rglob("*"):
                            if path.is_file() and path!=output and not path.name.endswith(".partial"):
                                archive.write(path,prefix+"/"+path.relative_to(directory).as_posix())
            with zipfile.ZipFile(output) as archive:
                if archive.testzip():raise IOError("Output ZIP read-back failed")
            with output.open("rb") as handle:checksum=hashlib.file_digest(handle,"sha256").hexdigest()
            (attempt/"outputs.sha256").write_text(checksum+"  outputs.zip\n")
            result["persistent_readback"]="PASS"
            if AUTO_DOWNLOAD:
                try:files.download(str(output));result["download_request"]="API_RETURNED"
                except Exception as error:result["download_request"]="FAILED: "+str(error)
    except Exception as error:result["export_error"]=str(error)
    finally:
        result["runtime_release"]="REQUESTED_NOT_CONFIRMED" if AUTO_RELEASE else "DISABLED"
        if mounted:
            try:(attempt/"final_report.json").write_text(json.dumps(result,indent=2))
            except Exception:pass
        print(json.dumps(result,indent=2))
        if AUTO_RELEASE:
            try:runtime.unassign()
            except Exception as error:print("Runtime release request failed:",error)
